In [1]:
import polars as pl
import numpy as np
import joblib
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import accuracy_score, precision_score, recall_score
from xgboost import XGBClassifier

data = {
    "StudyHours": [2.5, 5.0, 5.5, 8.0, 1.0, 4.0, 6.0, 3.0, 7.5, 9.0],
    "Attendance": [70, 85, 60, 95, 50, 75, 80, 65, 90, 98],
    "Gender": ["M", "F", "F", "M", "M", "F", "M", "F", "F", "M"],
    "City": ["Delhi", "Mumbai", "Delhi", "Pune", "Mumbai", "Delhi", "Pune", "Mumbai", "Delhi", "Pune"],
    "Pass": [0, 1, 0, 1, 0, 1, 1, 0, 1, 1]
}
df = pl.DataFrame(data)

df = df.with_columns([
    pl.col("StudyHours").fill_null(pl.col("StudyHours").median()),
    pl.col("Attendance").fill_null(pl.col("Attendance").median()),
    (pl.col("StudyHours") + pl.col("Attendance")).alias("Engagement")
])

le = LabelEncoder()
df = df.with_columns(pl.Series("Gender", le.fit_transform(df["Gender"].to_numpy())))
df = df.to_dummies(columns=["City"])

y = df["Pass"].to_numpy()
X_df = df.drop("Pass")
X = X_df.to_numpy()

selector = SelectKBest(score_func=f_classif, k=4)
X_selected = selector.fit_transform(X, y)

X_train, X_test, y_train, y_test = train_test_split(X_selected, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

models = {
    "Logistic Regression": LogisticRegression(),
    "Random Forest": RandomForestClassifier(random_state=42),
    "XGBoost": XGBClassifier(eval_metric='logloss', random_state=42)
}

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    print(f"{name} | Accuracy: {accuracy_score(y_test, preds):.2f} | Precision: {precision_score(y_test, preds, zero_division=0):.2f}")

rf_best = RandomForestClassifier(random_state=42)
grid = GridSearchCV(rf_best, {'n_estimators': [50, 100], 'max_depth': [3, 5]}, cv=2)
grid.fit(X_train, y_train)

best_model = grid.best_estimator_
joblib.dump(best_model, "model.pkl")

print(f"\nBest Params: {grid.best_params_}")
print("Model saved as model.pkl")

plt.bar(range(X_selected.shape[1]), best_model.feature_importances_)
plt.title("Feature Importance")
plt.show()

def predict_student(features_scaled):
    model = joblib.load("model.pkl")
    return model.predict(features_scaled)

print("Prediction Test:", predict_student(X_test[0:1]))

ModuleNotFoundError: No module named 'xgboost'

In [6]:
import polars as pl
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score
from xgboost import XGBClassifier
import joblib

In [ ]:
df = pl.read_csv("data.csv")
df

In [5]:
df = df.with_columns([
    pl.col("StudyHours").fill_null(pl.col("StudyHours").median()),
    pl.col("SleepHours").fill_null(pl.col("SleepHours").median()),
    pl.col("Attendance").fill_null(pl.col("Attendance").median()),
    pl.col("PreviousMarks").fill_null(pl.col("PreviousMarks").median())
])
df

ColumnNotFoundError: unable to find column "SleepHours"; valid columns: ["StudyHours", "Attendance", "Gender", "City_Delhi", "City_Mumbai", "City_Pune", "Pass", "Engagement"]

In [4]:
if "Gender" in df.columns:
    df = df.with_columns(
        pl.when(pl.col("Gender") == "Male").then(1).otherwise(0).alias("Gender")
    )
df

ComputeError: cannot compare string with numeric type (i64)

This error occurred in the following expression:
	[(col("Gender")) == ("Male")]
while evaluating this larger expression:
	.when([(col("Gender")) == ("Male")]).then(dyn int: 1).otherwise(dyn int: 0)
